In [1]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

import requests

In [2]:
from functions import read_api_key

In [4]:
# Read data from previous work on cleaning the data
yerevan_historical = pd.read_csv('../database/yerevan_historical_clean/sale_historical.csv')
yerevan_actual = pd.read_csv('../database/yerevan_actual_clean/sale_actual.csv')

## Modifying historical data columns

In [5]:
# Dropinng unnecessary columns and adding column for merging
yerevan_actual['backup_status'] = 'not sold'
yerevan_historical.drop(columns=[ 'days_in'], inplace=True)

In [6]:
df = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [7]:
# Dropping unnecessary columns
df.drop(columns=['links', 'source', 'rent_or_sale'], inplace=True)

### Removing houses that were wrongly classified as sold

In [11]:
df[df['id']==10895]

,backup_status,id,price,rooms,square_meters,address,sold_date,furniture,renovation,price_per_meter,...,height,bathroom,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered
74485,not sold,10895,125000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",NaN,1.0,NaN,2314.814815,...,2.8,1.0,2023-03-13 18:12:05,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513104,40.166181


In [9]:
df = df.drop_duplicates(subset=['id'], keep='last')
df = df.loc[~((df['backup_status'] == 'sold') & (df['id'].duplicated()))]

In [10]:
df = df.dropna(subset=['district']).reset_index(drop=True)

### Categorizing Neighborhoods

as our data has x, y coordinates setup we wrote a script to generate the corresponding neighborhood for each house


this is later used for creating more interesting visuals in arcgis, and running the model with the context of the neighborhoods for each house

In [55]:
df2 = df.head(75732).reset_index(drop=True)

In [15]:
# function defined in functions.py
api_key = read_api_key()

addresses = []

In [33]:
while True:
    try:
        for index, row in df2.iterrows():
            latlng = f"{row['x']},{row['y']}"
            # send a request to the Google Maps API and get most specific 'APPROXIMATE' response
            url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latlng}&location_type=APPROXIMATE&key={api_key}"
            response = requests.get(url)
            data = response.json()
            address = data['results'][0]['address_components'][0]['long_name']
            df2.drop(index=df2.index[0], axis=0, inplace=True)
            addresses.append(address)
    except:
        # if there was an error, print a message and try again
        print("An error occurred. Retrying...")
        continue
    else:
        # if there was no error, break out of the loop and continue with the rest of the code
        break

An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error

In [87]:
df['neighborhood'] = addresses

took postal code for a few of the houses so we replaced the values to their corresponding neighborhoods

In [ ]:
df.loc[df['neighborhood'] == '0054', 'neighborhood'] = 'Davtashen Block'
df.loc[df['neighborhood'] == '0059', 'neighborhood'] = 'Nor Nork 9th Microdistrict'

df['neighborhood'] = df['neighborhood'].str.replace(' ', '_')
df['neighborhood'] = df['neighborhood'].str.replace('-', '_')

In [87]:
# writing to csv
df.to_csv(r'../database/yerevan_combined_clean/sale_combined_neighborhood.csv', index=False)

small amount of rows weren't classified so we remove them

In [88]:
df = pd.read_csv('../database/yerevan_combined_clean/sale_combined_neighborhood.csv')

In [182]:
df = df.dropna(subset=['neighborhood'])

In [184]:
# seperating the dataframes from histroical and actual
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

we have to remove neighborhoods that have a count lower than so that when we run models

that is only in the context of each neighborhood we don't get errors

In [186]:
# removing neighborhoods with low counts for historical
neighborhood_counts = sale_historical['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_historical = sale_historical[~sale_historical['neighborhood'].isin(neighborhoods_to_remove)]

In [187]:
# removing neighborhoods with low counts for actual
neighborhood_counts = sale_actual['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_actual = sale_actual[~sale_actual['neighborhood'].isin(neighborhoods_to_remove)]

In [196]:
# not needed for sale_actual as all nans
sale_actual.drop(columns=['sold_date'], inplace=True)

In [199]:
# Only taking date for sold_date
sale_historical['sold_date'] = pd.to_datetime(sale_historical['sold_date'])

# Extract the date part from the datetime column
sale_historical['sold_date'] = sale_historical['sold_date'].dt.date

In [200]:
# populate to csv
sale_historical.to_csv(r'../database/yerevan_combined_clean/historical_sold.csv', index=False)
sale_actual.to_csv(r'../database/yerevan_combined_clean/actual_not_sold.csv', index=False)

the following version of the sale_historical dataset is optimized to run GLR model in ARCGIS by dropping the missing values and then making sure

we have enough degrees of freedom to run GLR because of too few features or too many explanatory variables.

In [ ]:
houses_filt = sale_historical.dropna(subset=['square_meters', 'height', 'walking_distance_to_metro(m)', 'bathroom'])

neighborhood_counts = houses_filt['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 9]['neighborhood'].tolist()
houses_filt = houses_filt[~houses_filt['neighborhood'].isin(neighborhoods_to_remove)]

In [ ]:
houses_filt.to_csv(r'../database/yerevan_combined_clean/historical_optimized_glr.csv', index=False)

In [3]:
sale_actual = pd.read_csv(r'../database/yerevan_combined_clean/actual_not_sold.csv')

In [15]:
sale_historical_glr = pd.read_csv(r'../database/yerevan_combined_clean/historical_optimized_glr.csv')

In [21]:
sale_actual_glr = sale_actual.dropna(subset=['square_meters', 'height', 'walking_distance_to_metro(m)', 'bathroom'])

neighborhood_counts = sale_actual_glr['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 11]['neighborhood'].tolist()
sale_actual_glr = sale_actual_glr[~sale_actual_glr['neighborhood'].isin(neighborhoods_to_remove)]

In [27]:
sale_actual_glr.groupby('neighborhood')['walking_distance_to_metro(m)'].nunique()

neighborhood
15th_Quartier                  10
1st_Block                      13
2nd_Block                      17
3rd_Block                       7
4th_Block                      15
8th_Quarter                     8
A2                              9
A3                              9
Ajapniak                       36
Antarayin                      21
Arabkir                        26
Araratian                      20
Avan                           22
Avan_Arinj                     18
Aygedzor                       17
Aygestan                        8
B1                             13
B2                             12
Charents                        3
Davtashen_Block                31
Duryan                          1
Erebuni                        31
Kanaker                        22
Kanaker_Zeytun                 43
Kentron                       327
Kond                           11
Lukashin                        4
Malatia_Sebastia                5
Narekatsi                       3
N

In [25]:
# verifying that sale actual is a subset of sale historical neighborhood values
sale_actual_glr['neighborhood'].isin(sale_historical_glr['neighborhood']).all()

True

In [ ]:
sale_actual_glr.to_csv(r'../database/yerevan_combined_clean/actual_optimized_glr.csv', index=False)